# Experiment 2A — Token Checker: Prompt Generation & Activation Extraction

**Purpose:** For every universal circuit that has a keyword token, build a "token checker" mask from prompts where the token appears *without* the concept.

For each testable object in `KEYWORD_MAP`, we:
1. Generate 50 validated Python prompts containing the keyword token but **not** the associated AST/builtin concept.
2. Feed each prompt through the CSP model and record MLP activations at all 8 layers.
3. Apply the same epsilon + consistency thresholds as Module 2 to produce a binary **token checker mask**.

The resulting masks are saved to `token_checker_masks.h5`. Notebook 5B compares these masks against the universal circuit masks to measure what fraction of each circuit is token-driven vs concept-driven.

In [ ]:
# Cell 1 – Configuration
import random
import numpy as np

random.seed(42)
np.random.seed(42)

EPSILON = 0.001
CONSISTENCY_THRESHOLD = 0.8
N_LAYERS = 8
N_TARGET = 50
OVERSHOOT_FACTOR = 3

print("Config OK")
print(f"  EPSILON = {EPSILON}")
print(f"  CONSISTENCY_THRESHOLD = {CONSISTENCY_THRESHOLD}")
print(f"  N_LAYERS = {N_LAYERS}")
print(f"  N_TARGET = {N_TARGET}")
print(f"  OVERSHOOT_FACTOR = {OVERSHOOT_FACTOR}")

In [ ]:
# Cell 2 – Imports and pip installs
import subprocess, sys

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "huggingface_hub"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, types, importlib.util, glob, inspect
import numpy as np
import pandas as pd
import torch
import h5py
from tqdm.auto import tqdm
from pathlib import Path

print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Cell 3 – Mount Drive / detect environment
import os, sys, shutil, subprocess

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = Path("/content/drive/MyDrive/DATA/CSP-Atlas")
    SRC_PATH = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/CSP-Atlas")
    SRC_PATH = "/Users/piotrwilam/Code/CSP-Atlas/src"

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

os.makedirs(DATA_DIR, exist_ok=True)

print(f"IN_COLAB = {IN_COLAB}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"SRC_PATH = {SRC_PATH}")

In [ ]:
# Cell 4 – circuit_sparsity injection + Load CSP model and tokenizer

# ── Locate real gpt.py & hook_utils.py ───────────────────────────────────
tm_base     = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/openai")
tm_gpt_hits = glob.glob(os.path.join(tm_base, "*/gpt.py"))

if tm_gpt_hits:
    gpt_path  = tm_gpt_hits[0]
    hook_path = os.path.join(os.path.dirname(gpt_path), "hook_utils.py")
    print(f"Using transformers_modules cache: {os.path.dirname(gpt_path)}")
else:
    from huggingface_hub import hf_hub_download
    gpt_path  = hf_hub_download("openai/circuit-sparsity", "gpt.py")
    hook_path = hf_hub_download("openai/circuit-sparsity", "hook_utils.py")
    print("Using hf_hub_download (first run)")

hf_dir = os.path.dirname(gpt_path)

cs_pkg = types.ModuleType("circuit_sparsity")
cs_pkg.__path__ = [hf_dir]
cs_pkg.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity"] = cs_pkg

hook_spec = importlib.util.spec_from_file_location("circuit_sparsity.hook_utils", hook_path)
hook_mod  = importlib.util.module_from_spec(hook_spec)
hook_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.hook_utils"] = hook_mod
hook_spec.loader.exec_module(hook_mod)
cs_pkg.hook_utils = hook_mod

gpt_spec = importlib.util.spec_from_file_location("circuit_sparsity.gpt", gpt_path)
gpt_mod  = importlib.util.module_from_spec(gpt_spec)
gpt_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.gpt"] = gpt_mod
gpt_spec.loader.exec_module(gpt_mod)

_RealGPTConfig = gpt_mod.GPTConfig
_valid_params  = set(inspect.signature(_RealGPTConfig.__init__).parameters) - {"self"}

class _PatchedGPTConfig(_RealGPTConfig):
    def __init__(self, **kwargs):
        super().__init__(**{k: v for k, v in kwargs.items() if k in _valid_params})

gpt_mod.GPTConfig = _PatchedGPTConfig
cs_pkg.gpt = gpt_mod

print(f"circuit_sparsity ready — GPTConfig patched")

# ── Load model and tokenizer ─────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "openai/circuit-sparsity"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=torch.float16,
).to(DEVICE).eval()

print(f"Model loaded on {DEVICE}")

In [ ]:
# Cell 5 – KEYWORD_MAP definition (Section 1.1)
# Key = universal circuit name (as in HDF5)
# Value = dict with:
#   "keyword": the token string to embed in prompts
#   "forbidden_nodes": AST node types that must NOT appear in the parse tree
#   "forbidden_names": (builtins only) identifiers that must not appear as calls/bare refs

KEYWORD_MAP = {
    # --- AST nodes with dedicated keywords ---
    "ast__Import":       {"keyword": "import",   "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__ImportFrom":   {"keyword": "from",     "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__Break":        {"keyword": "break",    "forbidden_nodes": ["Break"]},
    "ast__Pass":         {"keyword": "pass",     "forbidden_nodes": ["Pass"]},
    "ast__Continue":     {"keyword": "continue", "forbidden_nodes": ["Continue"]},
    "ast__Assert":       {"keyword": "assert",   "forbidden_nodes": ["Assert"]},
    "ast__If":           {"keyword": "if",       "forbidden_nodes": ["If", "IfExp"]},
    "ast__While":        {"keyword": "while",    "forbidden_nodes": ["While"]},
    "ast__For":          {"keyword": "for",      "forbidden_nodes": ["For", "AsyncFor", "ListComp", "SetComp", "DictComp", "GeneratorExp"]},
    "ast__Return":       {"keyword": "return",   "forbidden_nodes": ["Return"]},
    "ast__With":         {"keyword": "with",     "forbidden_nodes": ["With", "AsyncWith"]},
    "ast__Raise":        {"keyword": "raise",    "forbidden_nodes": ["Raise"]},
    "ast__Delete":       {"keyword": "del",      "forbidden_nodes": ["Delete"]},
    "ast__Global":       {"keyword": "global",   "forbidden_nodes": ["Global"]},
    "ast__Nonlocal":     {"keyword": "nonlocal", "forbidden_nodes": ["Nonlocal"]},
    "ast__Lambda":       {"keyword": "lambda",   "forbidden_nodes": ["Lambda"]},
    "ast__Yield":        {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__YieldFrom":    {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__Try":          {"keyword": "try",      "forbidden_nodes": ["Try"]},
    "ast__ClassDef":     {"keyword": "class",    "forbidden_nodes": ["ClassDef"]},
    "ast__FunctionDef":  {"keyword": "def",      "forbidden_nodes": ["FunctionDef", "AsyncFunctionDef"]},
    "ast__AsyncFor":     {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncFunctionDef": {"keyword": "async", "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncWith":    {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},

    # --- Builtins with short common-English tokens ---
    "builtin__list":     {"keyword": "list",     "forbidden_nodes": [],  "forbidden_names": ["list"]},
    "builtin__set":      {"keyword": "set",      "forbidden_nodes": [],  "forbidden_names": ["set"]},
    "builtin__map":      {"keyword": "map",      "forbidden_nodes": [],  "forbidden_names": ["map"]},
    "builtin__open":     {"keyword": "open",     "forbidden_nodes": [],  "forbidden_names": ["open"]},
    "builtin__type":     {"keyword": "type",     "forbidden_nodes": [],  "forbidden_names": ["type"]},
    "builtin__hash":     {"keyword": "hash",     "forbidden_nodes": [],  "forbidden_names": ["hash"]},
    "builtin__id":       {"keyword": "id",       "forbidden_nodes": [],  "forbidden_names": ["id"]},
    "builtin__all":      {"keyword": "all",      "forbidden_nodes": [],  "forbidden_names": ["all"]},
    "builtin__any":      {"keyword": "any",      "forbidden_nodes": [],  "forbidden_names": ["any"]},
    "builtin__sum":      {"keyword": "sum",      "forbidden_nodes": [],  "forbidden_names": ["sum"]},
    "builtin__min":      {"keyword": "min",      "forbidden_nodes": [],  "forbidden_names": ["min"]},
    "builtin__max":      {"keyword": "max",      "forbidden_nodes": [],  "forbidden_names": ["max"]},
    "builtin__next":     {"keyword": "next",     "forbidden_nodes": [],  "forbidden_names": ["next"]},
    "builtin__input":    {"keyword": "input",    "forbidden_nodes": [],  "forbidden_names": ["input"]},
    "builtin__len":      {"keyword": "len",      "forbidden_nodes": [],  "forbidden_names": ["len"]},
    "builtin__range":    {"keyword": "range",    "forbidden_nodes": [],  "forbidden_names": ["range"]},
    "builtin__filter":   {"keyword": "filter",   "forbidden_nodes": [],  "forbidden_names": ["filter"]},
    "builtin__print":    {"keyword": "print",    "forbidden_nodes": [],  "forbidden_names": ["print"]},
    "builtin__int":      {"keyword": "int",      "forbidden_nodes": [],  "forbidden_names": ["int"]},
    "builtin__float":    {"keyword": "float",    "forbidden_nodes": [],  "forbidden_names": ["float"]},
    "builtin__str":      {"keyword": "str",      "forbidden_nodes": [],  "forbidden_names": ["str"]},
    "builtin__bool":     {"keyword": "bool",     "forbidden_nodes": [],  "forbidden_names": ["bool"]},
    "builtin__round":    {"keyword": "round",    "forbidden_nodes": [],  "forbidden_names": ["round"]},
    "builtin__zip":      {"keyword": "zip",      "forbidden_nodes": [],  "forbidden_names": ["zip"]},
    "builtin__sorted":   {"keyword": "sorted",   "forbidden_nodes": [],  "forbidden_names": ["sorted"]},
    "builtin__super":    {"keyword": "super",    "forbidden_nodes": [],  "forbidden_names": ["super"]},
    "builtin__iter":     {"keyword": "iter",     "forbidden_nodes": [],  "forbidden_names": ["iter"]},
    "builtin__object":   {"keyword": "object",   "forbidden_nodes": [],  "forbidden_names": ["object"]},
    "builtin__bytes":    {"keyword": "bytes",    "forbidden_nodes": [],  "forbidden_names": ["bytes"]},
    "builtin__dict":     {"keyword": "dict",     "forbidden_nodes": [],  "forbidden_names": ["dict"]},
    "builtin__tuple":    {"keyword": "tuple",    "forbidden_nodes": [],  "forbidden_names": ["tuple"]},
    "builtin__property": {"keyword": "property", "forbidden_nodes": [],  "forbidden_names": ["property"]},
    "builtin__complex":  {"keyword": "complex",  "forbidden_nodes": [],  "forbidden_names": ["complex"]},
    "builtin__reversed": {"keyword": "reversed", "forbidden_nodes": [],  "forbidden_names": ["reversed"]},
}

print(f"KEYWORD_MAP: {len(KEYWORD_MAP)} testable objects")
n_ast = sum(1 for k in KEYWORD_MAP if k.startswith("ast__"))
n_builtin = sum(1 for k in KEYWORD_MAP if k.startswith("builtin__"))
print(f"  AST nodes: {n_ast}")
print(f"  Builtins:  {n_builtin}")

In [ ]:
# Cell 6 – IDENTIFIER_VARIANTS definition (Section 2.3)
# Identifiers that contain the keyword as a substring but are NOT the keyword itself.
# Used for Category C prompts (variable/function names containing the keyword).

IDENTIFIER_VARIANTS = {
    "import":   ["important_data", "reimport_flag", "import_path_str"],
    "break":    ["breakdown_count", "breakpoint_flag", "unbreakable"],
    "pass":     ["passport_number", "password_hash", "bypass_mode"],
    "continue": ["continuous_mode", "discontinue_flag", "continuum"],
    "for":      ["format_string", "information", "formula_result", "before_update"],
    "if":       ["notification", "verification", "modification", "amplifier"],
    "while":    ["meanwhile_flag", "worthwhile", "awhile_counter"],
    "return":   ["return_value_str", "unreturnable", "return_label"],
    "assert":   ["assertive_mode", "reassert_count"],
    "with":     ["width_value", "withdraw_amount", "within_range"],
    "class":    ["classification", "classic_mode", "subclass_name"],
    "def":      ["default_value", "undefined_flag", "defiant_mode", "defense_level"],
    "yield":    ["yielded_count", "unyielding"],
    "try":      ["retry_count", "country_code", "entry_point"],
    "raise":    ["raised_flag", "fundraise_total", "praise_count"],
    "del":      ["delivery_date", "delta_value", "model_name", "delegate_to"],
    "global":   ["global_str_ref", "globalization"],
    "nonlocal": ["nonlocal_ref_str"],
    "lambda":   ["lambda_str_ref"],
    "async":    ["async_str_label"],
    # Builtins
    "list":     ["checklist_items", "listing_data", "blacklist_mode"],
    "set":      ["settings_config", "reset_flag", "offset_value", "sunset_time"],
    "map":      ["mapped_result", "roadmap_items", "bitmap_size"],
    "dict":     ["dictionary_words", "verdict_text", "dictation_mode"],
    "open":     ["opening_hours", "reopening", "openly_shared"],
    "type":     ["typewriter_mode", "archetype", "prototype_id"],
    "print":    ["printable_chars", "fingerprint_id", "blueprint_ref"],
    "int":      ["interval_ms", "painting_id", "intelligence", "pointer_val"],
    "float":    ["floating_label", "afloat_status"],
    "str":      ["string_data", "stream_id", "structure_type", "destroy_flag"],
    "range":    ["arrangement", "strange_value", "orange_count"],
    "sum":      ["summary_text", "consumer_id", "assumption"],
    "max":      ["maximum_str", "climax_point"],
    "min":      ["minimum_str", "admin_level", "minute_count", "dominion"],
    "len":      ["length_str", "lender_id", "silent_mode", "calendar_ref"],
    "hash":     ["hashtag_count", "rehash_data"],
    "bool":     ["boolean_str", "booleanize"],
    "round":    ["roundup_notes", "background_val", "surround_count"],
    "zip":      ["zipcode_val", "unzip_path"],
    "all":      ["alliance_name", "install_path", "wallpaper"],
    "any":      ["company_name", "many_items", "tyranny"],
    "next":     ["next_str_ref", "annex_id"],
    "input":    ["input_str_ref"],
    "filter":   ["filter_str_ref", "unfiltered_data"],
    "object":   ["objection_text", "objective_id"],
    "id":       ["idea_count", "identity_str", "video_ref", "oxide_level"],
    "super":    ["supervisor_name", "superfluous"],
    "iter":     ["literal_count", "iterator_str_ref"],
    "bytes":    ["bytes_str_ref", "gigabytes_val"],
    "sorted":   ["sorted_str_ref", "unsorted_data"],
    "tuple":    ["tuple_str_ref", "quintuple_val"],
    "property": ["property_str_ref"],
    "complex":  ["complexity_score", "complex_str_ref"],
    "reversed": ["reversed_str_ref"],
}

print(f"IDENTIFIER_VARIANTS: {len(IDENTIFIER_VARIANTS)} keywords covered")

In [ ]:
# Cell 7 – Word banks and startup validation (Section 2.3)

VAR_NAMES = [
    "data", "result", "value", "item", "record", "entry", "output",
    "signal", "status", "config", "mode", "level", "target", "source",
    "handler", "token", "state", "buffer", "counter", "index",
    "offset", "block", "phase", "cycle", "frame", "chunk", "batch",
    "queue", "score", "label", "flag", "limit", "depth", "width_val",
]

STRING_WORDS = [
    "waiting", "ready", "starting", "completed", "processing",
    "running", "loading", "checking", "updating", "building",
    "reading", "writing", "sending", "receiving", "tracking",
    "reference", "documentation", "deployment", "production",
    "analysis", "operation", "execution", "validation", "review",
    "schedule", "planning", "progress", "workflow", "pipeline",
    "resource", "component", "structure", "algorithm", "protocol",
    "standard", "baseline", "threshold", "interval", "duration",
    "capacity", "frequency", "priority", "sequence", "pattern",
    "segment", "fragment", "section", "portion", "division",
]

CONTEXT_WRAPPERS = [
    "{snippet}",
    "x = 1\n{snippet}\ny = 2",
    "data = []\n{snippet}\nresult = None",
    "def func():\n    {snippet}\n    return None",
    "def process():\n    {snippet}",
    "class Obj:\n    def method(self):\n        {snippet}",
]

# Remove any var names that contain a testable keyword
all_keywords = set(v["keyword"] for v in KEYWORD_MAP.values())
safe_vars = [v for v in VAR_NAMES if not any(kw in v for kw in all_keywords)]
print(f"VAR_NAMES: {len(VAR_NAMES)} -> {len(safe_vars)} after keyword filtering")
VAR_NAMES = safe_vars

print(f"STRING_WORDS: {len(STRING_WORDS)}")
print(f"CONTEXT_WRAPPERS: {len(CONTEXT_WRAPPERS)}")

In [ ]:
# Cell 8 – Validation functions (Sections 3.1–3.4)
import ast as ast_module


def get_all_ast_info(code_string):
    """Return set of all AST node type names and all Name identifiers."""
    try:
        tree = ast_module.parse(code_string)
    except SyntaxError:
        return None, None

    node_types = set()
    name_ids = set()  # identifiers used as Name nodes
    call_names = set()  # identifiers used as function calls

    for node in ast_module.walk(tree):
        node_types.add(type(node).__name__)
        if isinstance(node, ast_module.Name):
            name_ids.add(node.id)
        if isinstance(node, ast_module.Call):
            if isinstance(node.func, ast_module.Name):
                call_names.add(node.func.id)

    return node_types, call_names


def check_concept_absent(code_string, obj_key):
    """
    Verify the target concept is NOT present in the parse tree.
    """
    info = KEYWORD_MAP[obj_key]
    node_types, call_names = get_all_ast_info(code_string)

    if node_types is None:
        return False  # parse failed

    # Check forbidden AST nodes
    for forbidden in info["forbidden_nodes"]:
        if forbidden in node_types:
            return False

    # Check forbidden builtin names (for builtins only)
    if "forbidden_names" in info:
        for forbidden_name in info["forbidden_names"]:
            if forbidden_name in call_names:
                return False

    return True


def check_token_present(code_string, keyword, tokenizer):
    """
    Verify the keyword appears as a token in the tokenizer output.

    We check if any individual token decodes to exactly the keyword,
    OR if the keyword appears as a substring in the decoded token
    (for cases where the tokenizer keeps it as part of a larger token
    inside a string).
    """
    token_ids = tokenizer.encode(code_string, add_special_tokens=False)

    # Method 1: exact match
    keyword_token_ids = tokenizer.encode(keyword, add_special_tokens=False)
    # Check if keyword_token_ids appears as a contiguous subsequence
    ids_list = token_ids
    kw_len = len(keyword_token_ids)
    for i in range(len(ids_list) - kw_len + 1):
        if ids_list[i:i+kw_len] == keyword_token_ids:
            return True

    # Method 2: decoded substring match (fallback)
    for tid in token_ids:
        decoded = tokenizer.decode([tid]).strip()
        if decoded == keyword:
            return True

    return False


def validate_prompt(code_string, obj_key, tokenizer):
    """Returns True if prompt is valid for token-without-concept testing."""
    keyword = KEYWORD_MAP[obj_key]["keyword"]

    # 1. Must parse
    node_types, call_names = get_all_ast_info(code_string)
    if node_types is None:
        return False

    # 2. Concept must be absent
    if not check_concept_absent(code_string, obj_key):
        return False

    # 3. Token must be present
    if not check_token_present(code_string, keyword, tokenizer):
        return False

    return True


print("Validation functions defined: get_all_ast_info, check_concept_absent, check_token_present, validate_prompt")

In [ ]:
# Cell 9 – Generation functions (Sections 2.4–2.5)
import random


def generate_raw_prompts(keyword, n_per_category=10):
    """
    Generate candidate prompts for one keyword across all 5 categories.
    Returns list of (prompt_text, category_label) tuples.
    No validation yet — that happens in validate_prompt.
    """
    candidates = []

    for _ in range(n_per_category):
        w1, w2, w3 = random.sample(STRING_WORDS, 3)
        v1, v2 = random.sample(VAR_NAMES, 2)
        wrapper = random.choice(CONTEXT_WRAPPERS)

        # Category A — String literal
        snippet_a = random.choice([
            f'{v1} = "waiting {keyword} {w1}"',
            f'{v1} = "{w1} {keyword} {w2} reference"',
            f'{v1} = "the {keyword} of {w1}"',
            f'{v1} = "no {keyword} needed for {w1}"',
            f'{v1} = "{w1} {keyword} {w2}"',
        ])
        candidates.append((wrapper.format(snippet=snippet_a), "A_string"))

        # Category B — Comment
        snippet_b = random.choice([
            f'{v1} = 42  # {keyword} {w1} see docs',
            f'# {keyword} {w1} {w2} reference\n{v1} = 10',
            f'{v1} = []  # TODO {keyword} this later',
            f'# note: {keyword} not needed for {w1}\n{v1} = True',
        ])
        candidates.append((wrapper.format(snippet=snippet_b), "B_comment"))

        # Category C — Identifier
        ident_variants = IDENTIFIER_VARIANTS.get(keyword, [])
        if ident_variants:
            ident = random.choice(ident_variants)
            snippet_c = random.choice([
                f'{ident} = 42',
                f'{ident} = "{w1}"',
                f'{ident} = [{v1} for {v1} in []]' if keyword != "for" else f'{ident} = 42',
                f'{ident} = True',
            ])
            candidates.append((wrapper.format(snippet=snippet_c), "C_identifier"))

        # Category D — Dictionary key
        snippet_d = random.choice([
            f'{v1} = {{"{keyword}": True, "{w1}": 42}}',
            f'{v1} = {{"{w1}": 10, "mode": "{keyword}"}}',
            f'{v1} = {{"action": "{w1}", "{keyword}": "{w2}"}}',
        ])
        candidates.append((wrapper.format(snippet=snippet_d), "D_dictkey"))

        # Category E — Print call
        snippet_e = random.choice([
            f'print("{w1} {keyword} {w2}")',
            f'print("starting {keyword} {w1}")',
            f'print("{keyword} completed for {w1}")',
        ])
        candidates.append((wrapper.format(snippet=snippet_e), "E_print"))

    return candidates


def generate_for_keyword(obj_key, target_n=N_TARGET):
    """Generate and validate prompts for one object. Returns exactly target_n or fewer."""
    keyword = KEYWORD_MAP[obj_key]["keyword"]
    raw = generate_raw_prompts(keyword, n_per_category=target_n * OVERSHOOT_FACTOR // 5)
    valid = [p for p in raw if validate_prompt(p[0], obj_key, tokenizer)]

    # Balance categories: take up to target_n // 5 per category, fill remainder from any
    by_cat = {}
    for text, cat in valid:
        by_cat.setdefault(cat, []).append(text)

    per_cat = target_n // 5  # = 10
    selected = []
    for cat in ["A_string", "B_comment", "C_identifier", "D_dictkey", "E_print"]:
        selected.extend(by_cat.get(cat, [])[:per_cat])

    # If short (e.g. C_identifier had few valid), fill from others
    if len(selected) < target_n:
        remaining = [t for t, c in valid if t not in selected]
        selected.extend(remaining[:target_n - len(selected)])

    return selected[:target_n]


print("Generation functions defined: generate_raw_prompts, generate_for_keyword")

In [ ]:
# Cell 10 – Main generation loop
all_validated_prompts = {}
prompt_counts = {}

for obj_key in tqdm(sorted(KEYWORD_MAP.keys()), desc="Generating prompts"):
    keyword = KEYWORD_MAP[obj_key]["keyword"]
    prompts = generate_for_keyword(obj_key, target_n=N_TARGET)
    all_validated_prompts[obj_key] = prompts
    prompt_counts[obj_key] = len(prompts)

    status = "OK" if len(prompts) >= N_TARGET else f"SHORT ({len(prompts)})"
    print(f"  {obj_key}: {len(prompts)} valid prompts [{status}]")

total = sum(prompt_counts.values())
print(f"\nTotal: {total} prompts across {len(all_validated_prompts)} objects")
short = [k for k, v in prompt_counts.items() if v < N_TARGET]
if short:
    print(f"WARNING: {len(short)} objects have < {N_TARGET} prompts: {short}")

In [ ]:
# Cell 11 – ActivationExtractor setup
import re
from module2.extraction import ActivationExtractor

# Auto-detect MLP pattern
mlp_pattern = None
seen_lids = set()
for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        lid = int(m.group(2))
        seen_lids.add(lid)
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)

print(f"Pattern: {mlp_pattern} | Layers: {sorted(seen_lids)}")

extractor = ActivationExtractor(
    model=model, tokenizer=tokenizer, device=DEVICE,
    n_layers=N_LAYERS, use_hook_recorder=False,
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

# Sanity check
sample_prompt = 'x = "hello world"'
acts = extractor.extract(sample_prompt)
for lid, vec in sorted(acts.items()):
    print(f"  Layer {lid}: shape={tuple(vec.shape)}, mean |act|={vec.abs().mean().item():.6f}")

In [ ]:
# Cell 12 – Token checker mask extraction loop
from module2.binarization import RawActivationCollector

collector = RawActivationCollector(n_layers=N_LAYERS)
token_checker_masks = {}

for obj_key in tqdm(sorted(all_validated_prompts.keys()), desc="Extracting"):
    prompts = all_validated_prompts[obj_key]
    if len(prompts) == 0:
        print(f"  {obj_key}: SKIPPED (no valid prompts)")
        continue

    raw = collector.collect(extractor, prompts)
    n_prompts = raw["n_prompts"]

    masks = {}
    for lid, layer_data in raw["layers"].items():
        mean_act = layer_data["activation_sum"] / n_prompts
        firing_rate = layer_data["firing_count"].astype(np.float32) / n_prompts
        mask = (mean_act > EPSILON) & (firing_rate >= CONSISTENCY_THRESHOLD)
        masks[lid] = mask

    token_checker_masks[obj_key] = masks
    size_l4 = masks.get(4, np.zeros(1)).sum()
    print(f"  {obj_key}: {n_prompts} prompts, layer 4 mask size = {size_l4}")

print(f"\nExtracted masks for {len(token_checker_masks)} objects")

In [ ]:
# Cell 13 – Save token_checker_masks.h5 (Section 4.3)
OUTPUT_FILE = DATA_DIR / "token_checker_masks.h5"

with h5py.File(OUTPUT_FILE, "w") as f:
    tcm = f.create_group("token_checker_masks")

    for obj_key, layers in token_checker_masks.items():
        for layer_id, mask in layers.items():
            tcm.create_dataset(
                f"layer_{layer_id}/{obj_key}",
                data=mask.astype(np.bool_),
                compression="gzip"
            )

    # Metadata
    md = f.create_group("metadata")
    md.attrs["epsilon"] = EPSILON
    md.attrs["consistency_threshold"] = CONSISTENCY_THRESHOLD
    md.attrs["n_prompts_per_object"] = N_TARGET
    md.attrs["categories"] = "A_string,B_comment,C_identifier,D_dictkey,E_print"

    # Per-object stats
    stats = f.create_group("generation_stats")
    for obj_key in token_checker_masks:
        stats.attrs[f"{obj_key}_n_valid"] = prompt_counts[obj_key]

print(f"Saved: {OUTPUT_FILE}")

In [ ]:
# Cell 14 – Save prompts to parquet (Section 4.4)
PROMPTS_FILE = DATA_DIR / "token_checker_prompts.parquet"

records = []
for obj_key, prompts in all_validated_prompts.items():
    for i, text in enumerate(prompts):
        records.append({
            "object": obj_key,
            "keyword": KEYWORD_MAP[obj_key]["keyword"],
            "variation_id": i,
            "prompt_text": text,
        })

pd.DataFrame(records).to_parquet(PROMPTS_FILE)
print(f"Saved: {PROMPTS_FILE}")

In [ ]:
# Cell 15 – Cleanup and summary
extractor.remove_hooks()
print("Hooks removed.")
print(f"\n5A complete.")
print(f"  Objects tested: {len(token_checker_masks)}")
print(f"  Total prompts: {sum(prompt_counts.values())}")
print(f"  Output masks: {DATA_DIR / 'token_checker_masks.h5'}")
print(f"  Output prompts: {DATA_DIR / 'token_checker_prompts.parquet'}")